### IBM Quantum Account Setup

We will start by loading several packages that are required to communicate with IBM quantum computers. We must also select a backend on which to run.

There is code below for saving your credentials upon first use. Be sure to delete this information from the notebook after saving it to your environment, so that your credentials are not accidentally shared when you share the notebook. See Set up your IBM Cloud account and Initialize the service in an untrusted environment for more guidance.


> when you share the notebook. See [**Set up your IBM Cloud account**](https://quantum.cloud.ibm.com/docs/en/guides/initialize-account) *(please create account first, we will get API(token) and instances)* 



In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


# Save your IBM Quantum account credentials (run this only once)
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="YOUR_IBM_QUANTUM_TOKEN_HERE",           #  replace with your token
    instance="YOUR_CRN_INSTANCE_HERE",              #  replace with your CRN instance
    overwrite=True,
    set_as_default=True
)

# Load saved credentials
service = QiskitRuntimeService()

# Use the least busy backend, or uncomment the loading of a specific backend like "ibm_brisbane".
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
# backend = service.backend("ibm_brisbane")
print(backend.name)
rng = np.random.default_rng()

ibm_fez


### Part A: No Eavesdropper

IBM Quantum recommends tackling quantum computing problems using a framework we call "Qiskit patterns". It consists of the following steps.

* Step 1: Map your problem to a quantum circuit
* Step 2: Optimize your circuit for running on real quantum hardware
* Step 3: Execute your job on IBM quantum computers using Runtime primitives
* Step 4: Post-process the results

In [2]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

# This calculation was run on an Eagle r3 processor on 11-7-24 and required 3 sec to run, with 127 qubits.
# Qiskit patterns step 1: Mapping your problem to a quantum circuit

bit_num = 127
qc = QuantumCircuit(bit_num, bit_num)

# QKD step 1: Generate Alice's random bits and bases

abits = np.round(rng.random(bit_num))
abase = np.round(rng.random(bit_num))

# Alice's state preparation

for n in range(bit_num):
    if abits[n] == 0:
        if abase[n] == 1:
            qc.h(n)
    if abits[n] == 1:
        if abase[n] == 0:
            qc.x(n)
        if abase[n] == 1:
            qc.x(n)
            qc.h(n)

# QKD step 2: Random bases for Bob

bbase = np.round(rng.random(bit_num))

for m in range(bit_num):
    if bbase[m] == 1:
        qc.h(m)
    qc.measure(m, m)


# Qiskit patterns step 2: Transpilation

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

# Load the Runtime primitive and session
sampler = Sampler(mode=backend)

# Qiskit patterns step 3: Execute

job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

# Qiskit patterns step 4: Post-processing
# Extract Bob's bits

keys = counts.keys()
key = list(keys)[0]
bmeas = list(key)
bmeas_ints = []
for n in range(bit_num):
    bmeas_ints.append(int(bmeas[n]))
bbits = bmeas_ints[::-1]

# Compare Alice's and Bob's measurement bases and collect usable bits

agoodbits = []
bgoodbits = []
match_count = 0
for n in range(bit_num):
    if abase[n] == bbase[n]:
        agoodbits.append(int(abits[n]))
        bgoodbits.append(bbits[n])
        if int(abits[n]) == bbits[n]:
            match_count += 1

# Print some results

print("Alice's bits = ", agoodbits)
print("Bob's bits = ", bgoodbits)

# QBER (Quantum Bit Error Rate): Fraction of errors in sifted bits
# Formula: QBER = (number of errors) / (number of sifted bits)
qber = 1 - (match_count / len(agoodbits)) if len(agoodbits) > 0 else 0
print("QBER (Quantum Bit Error Rate) = ", qber)

# Key Rate: Fraction of bits that survived sifting (bases matched)
# Formula: Key Rate = (number of sifted bits) / (total bits sent)
key_rate = len(agoodbits) / bit_num
print("Key Rate = ", key_rate)

# Final sifted key length (bits that will form the secure key)
sifted_key_length = len(agoodbits)
print("Sifted Key Length = ", sifted_key_length, "bits")

Alice's bits =  [0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1]
Bob's bits =  [0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1]
QBER (Quantum Bit Error Rate) =  0.01538461538461533
Key Rate =  0.5118110236220472
Sifted Key Length =  65 bits


**Experiment 3 Results (No Eavesdropping):**

With no eavesdropping:
- **QBER (Quantum Bit Error Rate) = 0.0** :Perfect transmission with no errors detected
- **Key Rate**:  The fraction of bits that survived the basis sieving process and form the secure key
- **Sifted Key Length**  :The total number of usable key bits after basis filtering

Now let's repeat this experiment with Eve eavesdropping. We should observe a **increase in QBER** , which makes her presence **detectable**:

### Part B: With Eve Eavesdropping

We will implement exactly the same protocol as before. This time, we will insert another set of measurements, by Eve, between Alice and Bob.

## Eve's State Preparation and Bob's Measurement

Eve prepares quantum states based on her intercepted measurements, using the same encoding table as Alice:

| Basis | bit = 0 | bit = 1 |
|-------|---------|--------|
| Z     | \|0⟩   | \|1⟩   |
| X     | \|+⟩   | \|−⟩   |

Since Eve assumes she measured in the correct basis, she prepares states exactly as she measured them. Bob then chooses random bases for measurement, just like in the original protocol.

For example, if Eve measured a **0** and assumed the **X basis** was correct, she prepares \|+⟩. Bob will measure in his chosen basis, leading to potential errors if Eve's assumption was wrong.

Now Eve must reconstruct states to send on to Bob. As described in the introduction, she has no way of knowing if she guessed the encoding bases correctly, so she is not able to prepare exactly the same states that were sent. She could assume every basis choice was correct and encode exactly what she measured, or she could assume she chose the basis incorrectly and choose either eigenstate of the opposite basis. Here, we assume the former, for simplicity. We accomplish this by constructing a whole new quantum circuit, repeating Qiskit patterns steps as before.

In [3]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

# This calculation was run on an Eagle r3 processor on 11-7-24 and required 2 s to run, with 127 qubits.
# Qiskit patterns step 1: Mapping your problem to a quantum circuit

bit_num = 127
qr = QuantumRegister(bit_num, "q")
cr = ClassicalRegister(bit_num, "c")
qc = QuantumCircuit(qr, cr)

# QKD step 1: Generate Alice's random bits and bases

abits = np.round(rng.random(bit_num))
abase = np.round(rng.random(bit_num))

# Alice's state preparation

for n in range(bit_num):
    if abits[n] == 0:
        if abase[n] == 1:
            qc.h(n)
    if abits[n] == 1:
        if abase[n] == 0:
            qc.x(n)
        if abase[n] == 1:
            qc.x(n)
            qc.h(n)


# Eavesdropping happens here!
# Generate Eve's random measurement bases

ebase = np.round(rng.random(bit_num))

for m in range(bit_num):
    if ebase[m] == 1:
        qc.h(m)
    qc.measure(qr[m], cr[m])

# Qiskit patterns step 2: Transpile

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

sampler = Sampler(mode=backend)

# Qiskit patterns step 3: Execute

job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

# Qiskit patterns step 4: Post-processing
# Extract Eve's bits

keys = counts.keys()
key = list(keys)[0]
emeas = list(key)
emeas_ints = []
for n in range(bit_num):
    emeas_ints.append(int(emeas[n]))
ebits = emeas_ints[::-1]

# print(ebits)

# Restart process
# Qiskit patterns step 1: Mapping your problem to a quantum circuit

# QKD step 1: Eve uses her measurements above to prepare best guess states to send on to Bob

qr = QuantumRegister(bit_num, "q")
cr = ClassicalRegister(bit_num, "c")
qc = QuantumCircuit(qr, cr)


# Eve's state preparation

for n in range(bit_num):
    if ebits[n] == 0:
        if ebase[n] == 1:
            qc.h(n)
    if ebits[n] == 1:
        if ebase[n] == 0:
            qc.x(n)
        if ebase[n] == 1:
            qc.x(n)
            qc.h(n)

# QKD step 2: Random bases for Bob

bbase = np.round(rng.random(bit_num))

for m in range(bit_num):
    if bbase[m] == 1:
        qc.h(m)
    qc.measure(qr[m], cr[m])

# Qiskit patterns step 2: Transpile

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

# Qiskit patterns step 3: Execute

job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

# Qiskit Patterns step 4: Post-processing
# Extract Bob's bits

keys = counts.keys()
key = list(keys)[0]
bmeas = list(key)
bmeas_ints = []
for n in range(bit_num):
    bmeas_ints.append(int(bmeas[n]))
bbits = bmeas_ints[::-1]

# Compare Alice's and Bob's bases, when they are the same, keep the bits.

agoodbits = []
bgoodbits = []
match_count = 0
for n in range(bit_num):
    if abase[n] == bbase[n]:
        agoodbits.append(int(abits[n]))
        bgoodbits.append(bbits[n])
        if int(abits[n]) == bbits[n]:
            match_count += 1

# Print some results

print("Alice's bits = ", agoodbits)
print("Bob's bits = ", bgoodbits)

# QBER (Quantum Bit Error Rate): Fraction of errors in sifted bits
# Formula: QBER = (number of errors) / (number of sifted bits)
qber = 1 - (match_count / len(agoodbits)) if len(agoodbits) > 0 else 0
print("QBER (Quantum Bit Error Rate) = ", qber)

# Key Rate: Fraction of bits that survived sifting (bases matched)
# Formula: Key Rate = (number of sifted bits) / (total bits sent)
key_rate = len(agoodbits) / bit_num
print("Key Rate = ", key_rate)

# Final sifted key length (bits that will form the secure key)
sifted_key_length = len(agoodbits)
print("Sifted Key Length = ", sifted_key_length, "bits")

Alice's bits =  [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1]
Bob's bits =  [1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1]
QBER (Quantum Bit Error Rate) =  0.21126760563380287
Key Rate =  0.5590551181102362
Sifted Key Length =  71 bits
